# Landing Gear / Terrain Collision Validation

Self-contained validation notebook for the SUAS landing gear model and terrain collision
model.  No Godot, no live sim, no terrain data download required.

**Scenario:** small UAS flying a *stabilized* approach to flat terrain at 0 m WGS84 elevation.
Starting from 4 m AGL on a 6° glideslope at a realistic approach speed (Vref ≈ 1.2 × stall), the
autopilot holds the flight-path angle (n_z = cos γ feed-forward + proportional FPA correction) all
the way to touchdown — no flare, so it arrives at the approach sink rate (a firm touchdown at
~1.4 × stall). (This replaces the earlier fixed-`n_z`=1
approach, which did not actually fly the descent — it leveled off and skimmed in at ~2.7 × stall.)

**What this tests:**
- Main gear touches first at ~0.23 m AGL, nose gear at ~0.21 m AGL
- Strut spring/damper forces should balance aircraft weight at steady state
- The aircraft settles onto the gear during roll-out (does not float on the wing)
- Body collider hard constraint prevents terrain penetration
- Contact forces reported through Python binding match expected values

Run all cells in order (`Kernel → Restart & Run All`).

In [ ]:
# ── Imports and path setup ────────────────────────────────────────────────────
import os, sys, json, math, copy
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import HTML

# python/ directory is the parent of notebooks/
_PYTHON_DIR = Path('..').resolve()
if str(_PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(_PYTHON_DIR))

# The .pyd is compiled with MinGW and retains libwinpthread-1.dll as a dynamic
# dependency (brought in transitively via -lmingw32 from the SDL Conan package).
# Add the ucrt64 bin directory to the Windows DLL search path so the loader can
# find it without requiring ucrt64/bin on the system PATH.
if sys.platform == 'win32':
    os.add_dll_directory(r'C:\msys64\ucrt64\bin')

import liteaero_sim_py as sim
print(f'liteaero_sim_py loaded from {sim.__file__ if hasattr(sim, "__file__") else _PYTHON_DIR}')
print(f'Exported names: {[x for x in dir(sim) if not x.startswith("_")]}')

In [ ]:
# ── Aircraft configuration ─────────────────────────────────────────────────────
# Load the canonical SUAS config and patch the initial_state for the approach.

_REPO_ROOT  = _PYTHON_DIR.parent
_CONFIG_PATH = _REPO_ROOT / 'configs' / 'small_uas_ksba.json'

with open(_CONFIG_PATH) as f:
    base_cfg = json.load(f)

# Aircraft parameters from config
mass_kg  = base_cfg['inertia']['mass_kg']
S_ref_m2 = base_cfg['aircraft']['S_ref_m2']
cl_alpha = base_cfg['lift_curve']['cl_alpha']
cl_max   = base_cfg['lift_curve']['cl_max']
ar       = base_cfg['aircraft']['ar']
e        = base_cfg['aircraft']['e']
cd0      = base_cfg['aircraft']['cd0']

RHO_KGM3 = 1.225           # ISA sea-level air density
G_MPS2   = 9.80665
W_N      = mass_kg * G_MPS2
V_STALL_MPS = math.sqrt(2 * W_N / (RHO_KGM3 * S_ref_m2 * cl_max))

# ── Scenario parameters ────────────────────────────────────────────────────────
# A *flown* stabilized approach (not a fixed-load-factor controlled-flight-into-terrain):
# the aircraft holds the glideslope via flight-path-angle feedback (n_z = cos γ feed-forward
# plus a proportional FPA correction) ALL THE WAY to touchdown — no flare, so it arrives at the
# approach sink rate (a firm, realistic touchdown). Vref ≈ 1.2 × V_stall; from this short final
# the touchdown is ~1.4 × V_stall at the approach sink.
TERRAIN_ELEV_M   = 0.0     # flat terrain WGS84 elevation (m)
AGL_INIT_M       = 4.0     # initial altitude above terrain (m) — short final (low so it lands near Vref)
VREF_RATIO       = 1.20    # approach speed / stall speed (Vref)
V_APPROACH_MPS   = VREF_RATIO * V_STALL_MPS
GAMMA_DEG        = -6.0    # approach flight path angle (° — negative = descending)
HEADING_DEG      = 0.0     # heading (° true — 0 = north)
FPA_GAIN         = 2.0     # flight-path-angle-hold gain (Δn_z per rad of FPA error)
DT_S             = 0.01    # integration timestep (s) — 100 Hz
T_END_S          = 120.0   # simulation duration (s) — long enough to reach the low-speed tail

gamma_rad   = math.radians(GAMMA_DEG)
heading_rad = math.radians(HEADING_DEG)

q_inf_pa = 0.5 * RHO_KGM3 * V_APPROACH_MPS**2
qS       = q_inf_pa * S_ref_m2

# Trim lift coefficient and AoA at Vref on the glideslope (steady descent: n_z = cos γ)
CL_trim    = math.cos(gamma_rad) * W_N / qS
alpha_trim = CL_trim / cl_alpha   # rad (linear range)
pitch_init = alpha_trim + gamma_rad

# NED velocities (heading north, descending)
vN = V_APPROACH_MPS * math.cos(abs(gamma_rad)) * math.cos(heading_rad)
vE = V_APPROACH_MPS * math.cos(abs(gamma_rad)) * math.sin(heading_rad)
vD = V_APPROACH_MPS * math.sin(abs(gamma_rad))   # positive = downward in NED

# Patch initial_state — use equator/prime-meridian so ENU≈flat-Earth holds
approach_cfg = copy.deepcopy(base_cfg)
approach_cfg['initial_state'] = {
    'latitude_rad':       0.0,
    'longitude_rad':      0.0,
    'altitude_m':         TERRAIN_ELEV_M + AGL_INIT_M,
    'velocity_north_mps': vN,
    'velocity_east_mps':  vE,
    'velocity_down_mps':  vD,
    'wind_north_mps':     0.0,
    'wind_east_mps':      0.0,
    'wind_down_mps':      0.0,
    'pitch_rad':          pitch_init,
    'roll_rad':           0.0,
    'heading_rad':        heading_rad,
}

print(f'Aircraft mass       = {mass_kg:.1f} kg   weight = {W_N:.1f} N   V_stall = {V_STALL_MPS:.2f} m/s')
print(f'Approach Vref       = {V_APPROACH_MPS:.2f} m/s  ({VREF_RATIO:.2f}x stall)   gamma = {GAMMA_DEG:.1f} deg')
print(f'Trim CL             = {CL_trim:.3f}   alpha_trim = {math.degrees(alpha_trim):.2f} deg')
print(f'Initial pitch       = {math.degrees(pitch_init):.2f} deg')
print(f'NED velocity        = [{vN:.2f}, {vE:.2f}, {vD:.2f}] m/s')
print(f'Initial altitude    = {AGL_INIT_M:.1f} m AGL   (no flare — firm touchdown at the approach sink)')

In [ ]:
# ── Gear geometry (from config, for visualization reference) ──────────────────
gear_params = approach_cfg['landing_gear']['wheel_units']
GEAR_NAMES  = ['Nose', 'L-Main', 'R-Main']
GEAR_COLORS = ['tab:blue', 'tab:orange', 'tab:green']

# At zero deflection, contact patch altitude offset below CG (level aircraft):
#   offset = attach.z + tyre_radius   (body +Z = down)
for i, (g, name) in enumerate(zip(gear_params, GEAR_NAMES)):
    attach = g['attach_point_body_m']
    r      = g['tyre_radius_m']
    zmax   = g['travel_max_m']
    offset = attach[2] + r
    print(f'{name:8s}  attach={attach}  tyre_r={r:.2f} m  '
          f'travel_max={zmax:.2f} m  '
          f'first-contact AGL ≈ {offset:.3f} m  '
          f'k={g["spring_stiffness_npm"]:.0f} N/m')

# Static deflection at steady state (main gear pair carries ~all weight at low speed)
k_main   = gear_params[1]['spring_stiffness_npm']
delta_static = W_N / (2 * k_main)
print(f'\nExpected static main-gear deflection (each) ≈ {delta_static*1000:.1f} mm'
      f'  (W / (2·k) = {W_N:.1f} / {2*k_main:.0f})')

In [ ]:
# ── Simulation helper ──────────────────────────────────────────────────────────
def _run_sim(cfg, label=''):
    """Fly a stabilized FPA-hold approach + flare to touchdown; return result arrays."""
    terrain_local = sim.FlatTerrain(TERRAIN_ELEV_M)
    ac_local = sim.Aircraft(json.dumps(cfg), dt_s=DT_S)
    ac_local.set_terrain(terrain_local)

    cmd_local = sim.AircraftCommand()
    cmd_local.n_y               = 0.0
    cmd_local.roll_rate_wind_rps = 0.0
    cmd_local.throttle_nd       = 0.0          # idle approach

    N_local = int(T_END_S / DT_S)
    r = dict(
        t_s          = np.zeros(N_local),
        alt_m        = np.zeros(N_local),
        agl_m        = np.zeros(N_local),
        vel_ned      = np.zeros((N_local, 3)),
        vel_body     = np.zeros((N_local, 3)),
        accel_ned    = np.zeros((N_local, 3)),
        pitch_rad    = np.zeros(N_local),
        roll_rad_    = np.zeros(N_local),
        heading_rad_ = np.zeros(N_local),
        alpha_rad    = np.zeros(N_local),
        beta_rad     = np.zeros(N_local),
        p_rad_s      = np.zeros(N_local),
        q_rad_s      = np.zeros(N_local),
        r_rad_s      = np.zeros(N_local),
        airspeed     = np.zeros(N_local),
        cl_eff       = np.zeros(N_local),
        n_z_cmd      = np.zeros(N_local),
        cf_force     = np.zeros((N_local, 3)),
        cf_moment    = np.zeros((N_local, 3)),
        wow          = np.zeros(N_local, dtype=bool),
        strut_defl   = np.zeros((N_local, 3)),
        strut_rate   = np.zeros((N_local, 3)),
        wheel_spd    = np.zeros((N_local, 3)),
    )

    gamma_target = abs(gamma_rad)                  # descent-positive glideslope target

    for i in range(N_local):
        s  = ac_local.state()
        cf = ac_local.contact_forces()
        gs = ac_local.gear_strut_states()

        # ── Approach guidance: hold the glideslope to touchdown (no flare); hold on the ground ──
        if cf.weight_on_wheels:
            cmd_local.n_z = 1.0
        else:
            v_h = math.hypot(s.velocity_north_mps, s.velocity_east_mps)
            gamma_cur = math.atan2(s.velocity_down_mps, max(v_h, 0.1))
            cmd_local.n_z = max(0.2, min(1.8, math.cos(gamma_target) + FPA_GAIN * (gamma_cur - gamma_target)))

        r['t_s'][i]         = s.time_s
        r['alt_m'][i]       = s.altitude_m
        r['agl_m'][i]       = ac_local.agl_m()
        r['vel_ned'][i]     = [s.velocity_north_mps, s.velocity_east_mps, s.velocity_down_mps]
        r['vel_body'][i]    = s.velocity_body_mps
        r['accel_ned'][i]   = s.acceleration_ned_mps2
        r['pitch_rad'][i]   = s.pitch_rad
        r['roll_rad_'][i]   = s.roll_rad
        r['heading_rad_'][i]= s.heading_rad
        r['alpha_rad'][i]   = s.alpha_rad
        r['beta_rad'][i]    = s.beta_rad
        r['p_rad_s'][i]     = s.rates_body_rad_s[0]
        r['q_rad_s'][i]     = s.rates_body_rad_s[1]
        r['r_rad_s'][i]     = s.rates_body_rad_s[2]
        r['airspeed'][i]    = s.airspeed_m_s
        r['cl_eff'][i]      = ac_local.cl_eff
        r['n_z_cmd'][i]     = cmd_local.n_z
        r['cf_force'][i]    = cf.force_body_n
        r['cf_moment'][i]   = cf.moment_body_nm
        r['wow'][i]         = cf.weight_on_wheels
        if gs:
            for j, g in enumerate(gs):
                r['strut_defl'][i, j] = g.deflection_m
                r['strut_rate'][i, j] = g.deflection_rate_mps
                r['wheel_spd'][i, j]  = g.wheel_speed_rps

        ac_local.step(cmd_local, dt_s=DT_S, rho_kgm3=RHO_KGM3)

    td_idx = np.argmax(r['wow']) if np.any(r['wow']) else None
    r['td_t'] = r['t_s'][td_idx] if td_idx is not None else None
    r['label'] = label
    return r

# ── Run: with landing gear (primary) ──────────────────────────────────────────
results_gear    = _run_sim(approach_cfg,         label='With gear')

# ── Run: body-collider only (no landing gear) ─────────────────────────────────
_cfg_no_gear = copy.deepcopy(approach_cfg)
del _cfg_no_gear['landing_gear']
results_no_gear = _run_sim(_cfg_no_gear, label='No gear (body collider only)')

# ── Unpack primary (with-gear) results into module-level names used by cells below
t_s          = results_gear['t_s']
alt_m        = results_gear['alt_m']
agl_m        = results_gear['agl_m']
vel_ned      = results_gear['vel_ned']
vel_body     = results_gear['vel_body']
accel_ned    = results_gear['accel_ned']
pitch_rad    = results_gear['pitch_rad']
roll_rad_    = results_gear['roll_rad_']
heading_rad_ = results_gear['heading_rad_']
alpha_rad    = results_gear['alpha_rad']
beta_rad     = results_gear['beta_rad']
p_rad_s      = results_gear['p_rad_s']
q_rad_s      = results_gear['q_rad_s']
r_rad_s      = results_gear['r_rad_s']
airspeed     = results_gear['airspeed']
cl_eff       = results_gear['cl_eff']
cf_force     = results_gear['cf_force']
cf_moment    = results_gear['cf_moment']
wow          = results_gear['wow']
strut_defl   = results_gear['strut_defl']
strut_rate   = results_gear['strut_rate']
wheel_spd    = results_gear['wheel_spd']
td_t_main    = results_gear['td_t']

N = len(t_s)

# Derived: position integrated from NED velocity (relative to start)
north_m = np.cumsum(vel_ned[:, 0]) * DT_S
east_m  = np.cumsum(vel_ned[:, 1]) * DT_S

# Derived: aerodynamic forces. Lift uses the model's *effective* CL (cl_eff) — the actual
# aerodynamic CL applied — NOT a lift-curve reconstruction from the body-attitude alpha (which
# includes the gear-induced nose-down deviation on the ground and would imply phantom downforce).
k_ind   = 1.0 / (math.pi * ar * e)
CL_nom  = alpha_rad * cl_alpha          # nominal lift-curve value from body alpha (reference only)
CD      = cd0 + k_ind * cl_eff**2
q_dyn   = 0.5 * RHO_KGM3 * airspeed**2
L_N     = q_dyn * S_ref_m2 * cl_eff     # actual aero lift
D_N     = q_dyn * S_ref_m2 * CD

print(f'With-gear run:    {N} steps   first WoW at t = {td_t_main:.3f} s' if td_t_main else f'With-gear run: {N} steps   no WoW detected')
ng_td = results_no_gear['td_t']
print(f'No-gear run:      {N} steps   first WoW at t = {ng_td:.3f} s' if ng_td else f'No-gear run: {N} steps   no WoW detected')
if td_t_main is not None:
    td_i = int(np.argmax(wow))
    print(f'Touchdown speed   = {airspeed[td_i]:.2f} m/s  ({airspeed[td_i]/V_STALL_MPS:.2f}x stall)   '
          f'sink = {vel_ned[td_i, 2]:.2f} m/s')

In [ ]:
# ── Figure 0: Gear vs No-gear comparison ──────────────────────────────────────
_COLORS = {'With gear': 'tab:blue', 'No gear (body collider only)': 'tab:orange'}

fig0, axes0 = plt.subplots(2, 2, figsize=(13, 8))
fig0.suptitle('Gear vs No-gear — Key Metrics Comparison', fontsize=12, fontweight='bold')

for rr in [results_gear, results_no_gear]:
    col   = _COLORS[rr['label']]
    lbl   = rr['label']
    td_t  = rr['td_t']
    t     = rr['t_s']
    agl   = rr['agl_m']
    fz    = -rr['cf_force'][:, 2]   # upward contact force
    pitch = np.degrees(rr['pitch_rad'])
    wow_f = rr['wow'].astype(float)

    # AGL
    ax = axes0[0, 0]
    ax.plot(t, agl, color=col, label=lbl)

    # Contact force Fz vs weight
    ax = axes0[0, 1]
    ax.plot(t, fz, color=col, label=lbl)

    # Pitch angle
    ax = axes0[1, 0]
    ax.plot(t, pitch, color=col, label=lbl)

    # WoW
    ax = axes0[1, 1]
    ax.plot(t, wow_f, color=col, label=lbl, lw=1.2)
    if td_t is not None:
        for ax_ in axes0.flat:
            ax_.axvline(td_t, color=col, lw=0.8, ls=':', alpha=0.6)

# Decorate
ax = axes0[0, 0]
ax.axhline(0.0, color='saddlebrown', lw=1.5)
ax.set_xlabel('Time (s)'); ax.set_ylabel('AGL (m)')
ax.set_title('Height Above Ground (CG)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes0[0, 1]
ax.axhline(W_N, color='k', lw=1.0, ls='--', label=f'Weight {W_N:.1f} N')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Force (N)')
ax.set_title('Contact Force −Fz (upward reaction)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes0[1, 0]
ax.set_xlabel('Time (s)'); ax.set_ylabel('Pitch (°)')
ax.set_title('Pitch Angle'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes0[1, 1]
ax.set_ylim(-0.1, 1.3)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Weight-on-Wheels')
ax.set_title('Weight-on-Wheels Flag'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

fig0.tight_layout()
plt.show()

# ── Steady-state summary: both scenarios ──────────────────────────────────────
print()
for rr in [results_gear, results_no_gear]:
    lbl     = rr['label']
    t       = rr['t_s']
    ss_mask = (t > T_END_S - 2.0) & rr['wow']
    print(f'── {lbl} ──')
    if ss_mask.sum() < 5:
        print('  WARNING: < 5 WoW samples in last 2 s — aircraft not settled on ground.')
    else:
        mean_Fz   = rr['cf_force'][ss_mask, 2].mean()
        mean_agl  = rr['agl_m'][ss_mask].mean()
        fz_err    = abs((mean_Fz + W_N) / W_N) * 100
        mean_pitch = np.degrees(rr['pitch_rad'][ss_mask].mean())
        print(f'  Mean contact Fz  = {mean_Fz:.1f} N  (expected ≈ {-W_N:.1f} N)  error = {fz_err:.1f}%')
        print(f'  Mean AGL         = {mean_agl:.4f} m')
        print(f'  Mean pitch       = {mean_pitch:.2f}°')
        if fz_err > 5.0:
            print(f'  ⚠  Fz error {fz_err:.1f}% > 5%')
        if mean_agl < -0.01:
            print(f'  ⚠  Negative AGL — terrain penetration detected')
    print()

In [ ]:
# ── Helper: annotate touchdown on an axis ─────────────────────────────────────
def _annotate_td(ax, td_t, label='T/D', color='red', alpha=0.7):
    if td_t is None:
        return
    ax.axvline(td_t, color=color, lw=1.0, ls='--', alpha=alpha)
    ylim = ax.get_ylim()
    ax.text(td_t + 0.1, ylim[0] + 0.05*(ylim[1]-ylim[0]),
            label, color=color, fontsize=7, alpha=alpha)

In [ ]:
# ── Figure 1: Position and kinematics ─────────────────────────────────────────
fig1, axes1 = plt.subplots(3, 2, figsize=(13, 10))
fig1.suptitle('SUAS Landing Approach — Position and Kinematics', fontsize=12, fontweight='bold')

# Altitude
ax = axes1[0, 0]
ax.plot(t_s, alt_m, 'k')
ax.axhline(TERRAIN_ELEV_M, color='saddlebrown', lw=1.5, ls='-', label=f'terrain {TERRAIN_ELEV_M:.0f} m')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Altitude WGS84 (m)')
ax.set_title('CG Altitude'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# AGL
ax = axes1[0, 1]
ax.plot(t_s, agl_m, 'k')
ax.axhline(0.0, color='saddlebrown', lw=1.5)
gear_offsets = [p['attach_point_body_m'][2] + p['tyre_radius_m'] for p in gear_params]
for offset, name, col in zip(gear_offsets, GEAR_NAMES, GEAR_COLORS):
    ax.axhline(offset, color=col, lw=1, ls=':', alpha=0.7, label=f'{name} contact ≈{offset:.3f} m')
ax.set_xlabel('Time (s)'); ax.set_ylabel('AGL (m)')
ax.set_title('Height Above Ground Level (CG)')
ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# NED velocity
ax = axes1[1, 0]
ax.plot(t_s, vel_ned[:, 0], label='v_N')
ax.plot(t_s, vel_ned[:, 1], label='v_E')
ax.plot(t_s, vel_ned[:, 2], label='v_D (+ down)')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Velocity (m/s)')
ax.set_title('NED Velocity'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Body velocity
ax = axes1[1, 1]
ax.plot(t_s, vel_body[:, 0], label='u (fwd)')
ax.plot(t_s, vel_body[:, 1], label='v (right)')
ax.plot(t_s, vel_body[:, 2], label='w (down)')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Velocity (m/s)')
ax.set_title('Body-Frame Velocity'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Euler angles
ax = axes1[2, 0]
ax.plot(t_s, np.degrees(pitch_rad),    label='pitch θ')
ax.plot(t_s, np.degrees(roll_rad_),    label='roll φ')
ax.plot(t_s, np.degrees(alpha_rad),    label='α AoA', ls='--')
ax.plot(t_s, np.degrees(beta_rad),     label='β sideslip', ls='--')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Angle (°)')
ax.set_title('Euler Angles and Aerodynamic Angles'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Angular rates
ax = axes1[2, 1]
ax.plot(t_s, np.degrees(p_rad_s), label='p (roll)')
ax.plot(t_s, np.degrees(q_rad_s), label='q (pitch)')
ax.plot(t_s, np.degrees(r_rad_s), label='r (yaw)')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Rate (°/s)')
ax.set_title('Body Angular Rates'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

fig1.tight_layout()
plt.show()

In [ ]:
# ── Figure 2: Ground contact ───────────────────────────────────────────────────
fig2, axes2 = plt.subplots(3, 2, figsize=(13, 10))
fig2.suptitle('SUAS Landing Approach — Ground Contact', fontsize=12, fontweight='bold')

# WOW flag
ax = axes2[0, 0]
ax.fill_between(t_s, wow.astype(float), alpha=0.4, color='tab:red', label='WoW')
ax.plot(t_s, wow.astype(float), 'tab:red', lw=0.8)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Weight-on-Wheels')
ax.set_ylim(-0.1, 1.3); ax.set_title('Weight-on-Wheels Flag'); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Strut deflections
ax = axes2[0, 1]
for j, (name, col) in enumerate(zip(GEAR_NAMES, GEAR_COLORS)):
    ax.plot(t_s, strut_defl[:, j]*1000, color=col, label=name)
    ax.axhline(delta_static*1000, color=col, lw=0.8, ls=':', alpha=0.6)
ax.text(t_s[-1]*0.02, delta_static*1000 + 0.3, f'static δ_main={delta_static*1000:.1f} mm', fontsize=7, color='gray')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Deflection (mm)')
ax.set_title('Strut Deflection (compression)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Strut rates
ax = axes2[1, 0]
for j, (name, col) in enumerate(zip(GEAR_NAMES, GEAR_COLORS)):
    ax.plot(t_s, strut_rate[:, j]*1000, color=col, label=name)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Rate (mm/s)')
ax.set_title('Strut Compression Rate'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Wheel speed
ax = axes2[1, 1]
for j, (name, col) in enumerate(zip(GEAR_NAMES, GEAR_COLORS)):
    ax.plot(t_s, wheel_spd[:, j], color=col, label=name)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Wheel speed (rad/s)')
ax.set_title('Wheel Angular Speed'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Contact force body-Z (normal) — should approach W_N after settling
ax = axes2[2, 0]
ax.plot(t_s, -cf_force[:, 2], label='Fz body (up = −Fz_body)')
ax.axhline(W_N, color='k', lw=1.0, ls='--', label=f'Weight {W_N:.1f} N')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Force (N)')
ax.set_title('Contact Normal Force vs Aircraft Weight'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Contact force body-X and Y (longitudinal and lateral friction)
ax = axes2[2, 1]
ax.plot(t_s, cf_force[:, 0], label='Fx (fwd)')
ax.plot(t_s, cf_force[:, 1], label='Fy (right)')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Force (N)')
ax.set_title('Contact Friction Forces (body frame)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

fig2.tight_layout()
plt.show()

In [ ]:
# ── Figure 3: Contact moments and acceleration ─────────────────────────────────
fig3, axes3 = plt.subplots(2, 2, figsize=(13, 7))
fig3.suptitle('SUAS Landing Approach — Moments and Acceleration', fontsize=12, fontweight='bold')

# Contact moments body
ax = axes3[0, 0]
ax.plot(t_s, cf_moment[:, 0], label='Mx (roll)')
ax.plot(t_s, cf_moment[:, 1], label='My (pitch)')
ax.plot(t_s, cf_moment[:, 2], label='Mz (yaw)')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Moment (N·m)')
ax.set_title('Contact Moments (body frame)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# NED acceleration
ax = axes3[0, 1]
ax.plot(t_s, accel_ned[:, 0], label='aN')
ax.plot(t_s, accel_ned[:, 1], label='aE')
ax.plot(t_s, accel_ned[:, 2], label='aD (+down)')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Acceleration (m/s²)')
ax.set_title('NED Acceleration'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Aerodynamic coefficients. cl_eff is the model's actual applied CL; CL_nom is the lift-curve
# value at the body-attitude alpha (drops below cl_eff on the ground, where the gear pitches the
# body nose-down — applying CL_nom there would imply phantom downforce the model does not produce).
ax = axes3[1, 0]
ax.plot(t_s, cl_eff, label='C_L (effective, applied)')
ax.plot(t_s, CL_nom, label='C_L (nominal from body α)', ls=':')
ax.plot(t_s, CD, label='C_D')
ax.plot(t_s, cl_eff/np.maximum(CD, 1e-6), label='L/D', ls='--')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Coefficient / ratio')
ax.set_title('Lift/Drag Coefficients'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Lift and Drag forces (lift from the effective CL — the actual aerodynamic force)
ax = axes3[1, 1]
ax.plot(t_s, L_N, label='Lift L (N)')
ax.plot(t_s, D_N, label='Drag D (N)')
ax.axhline(W_N, color='k', lw=0.8, ls='--', label=f'Weight {W_N:.1f} N')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Force (N)')
ax.set_title('Aerodynamic Forces (lift from effective CL)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

fig3.tight_layout()
plt.show()

In [ ]:
# ── Wireframe animation geometry (data only) ──────────────────────────────────
# The render pipeline lives in the tools.anim_wireframe library (parallel frame
# rasterization); this cell only assembles the static geometry it consumes.

# Aircraft collider volumes in body frame (+X fwd, +Y right, +Z down):
#   (name, center_offset_body_m, half_extents_body_m, color)
VOL_DEFS = [
    ('fuselage', approach_cfg['body_collider']['volumes'][0]['center_offset_body_m'],
                 approach_cfg['body_collider']['volumes'][0]['half_extents_body_m'], 'gray'),
    ('wing',     approach_cfg['body_collider']['volumes'][1]['center_offset_body_m'],
                 approach_cfg['body_collider']['volumes'][1]['half_extents_body_m'], 'steelblue'),
    ('h_tail',   approach_cfg['body_collider']['volumes'][2]['center_offset_body_m'],
                 approach_cfg['body_collider']['volumes'][2]['half_extents_body_m'], 'steelblue'),
]

# Gear attach and tyre data
GEAR_ATTACH = np.array([g['attach_point_body_m'] for g in gear_params])   # (3,3)
GEAR_AXIS   = np.array([g['travel_axis_body']    for g in gear_params])   # (3,3)
GEAR_RADIUS = np.array([g['tyre_radius_m']       for g in gear_params])   # (3,)

# ENU trajectory (altitude above terrain)
cg_enu = np.column_stack([east_m, north_m, alt_m - TERRAIN_ELEV_M])

print(f'Geometry ready. Trajectory: {cg_enu.shape[0]} frames, '
      f'N range [{cg_enu[:,1].min():.1f}, {cg_enu[:,1].max():.1f}] m')


In [ ]:
# show every Nth simulation step (decimation for playback speed)
ANIM_DECIMATE = 10

# ── Wireframe animation — 4-view, parallel-rendered ───────────────────────────
# Side (North-Up), Top (East-North), Rear (East-Up), Isometric. Frames are
# decimated and rasterized across CPU cores by tools.anim_wireframe
# (ProcessPoolExecutor); the notebook is a single library call.

from tools import anim_wireframe

_anim_html = anim_wireframe.render_landing_animation(
    heading_rad=heading_rad_, pitch_rad=pitch_rad, roll_rad=roll_rad_,
    cg_enu=cg_enu, strut_defl=strut_defl, t_s=t_s, agl_m=agl_m,
    vol_defs=[(c, h, col) for (_, c, h, col) in VOL_DEFS],
    gear_attach=GEAR_ATTACH, gear_axis=GEAR_AXIS, gear_radius=GEAR_RADIUS,
    gear_colors=GEAR_COLORS, decimate=ANIM_DECIMATE, fps=25,
    ground_up_max=AGL_INIT_M + 1,
)
HTML(_anim_html)


In [ ]:
# ── Steady-state validation summary ───────────────────────────────────────────
# Check the last 2 seconds of simulation for steady-state values.

ss_mask = (t_s > T_END_S - 2.0) & wow
if ss_mask.sum() < 10:
    print('WARNING: fewer than 10 WoW samples in last 2 s — aircraft may not be settled.')
else:
    # Expected: contact force Fz (body) ≈ −W_N (body +Z = down, reaction is upward = negative)
    mean_Fz    = cf_force[ss_mask, 2].mean()
    mean_defl  = strut_defl[ss_mask].mean(axis=0)  # (3,)

    print('──── Steady-state validation (last 2 s on ground) ───────────────────')
    print(f'Aircraft weight W         = {W_N:.2f} N')
    print(f'Mean contact Fz body      = {mean_Fz:.2f} N  (expected ≈ {-W_N:.2f} N, i.e. upward)')
    print(f'Fz error                  = {mean_Fz - (-W_N):.3f} N  ({abs((mean_Fz + W_N)/W_N)*100:.1f}%)')
    print()
    print(f'Expected main-gear δ each = {delta_static*1000:.2f} mm  (W / (2·k_main))')
    for j, (name, d) in enumerate(zip(GEAR_NAMES, mean_defl)):
        print(f'  {name:8s}  mean δ = {d*1000:.2f} mm')

    print()
    mean_agl = agl_m[ss_mask].mean()
    print(f'Mean AGL (should be ≈ gear contact height): {mean_agl:.4f} m')

    # Flag potential bugs
    Fz_err_pct = abs((mean_Fz + W_N) / W_N) * 100
    if Fz_err_pct > 5.0:
        print(f'\n⚠  Contact Fz error {Fz_err_pct:.1f}% > 5% — possible gear spring or terrain-collision bug.')
    if mean_agl < -0.01:
        print(f'\n⚠  Negative AGL {mean_agl:.4f} m — terrain penetration: body collider or gear geometry bug.')
    if mean_agl > 0.30:
        print(f'\n⚠  AGL {mean_agl:.3f} m > 0.30 m at rest — aircraft not settling to ground: gear spring bug.')

## Touch-and-Go Validation

Same 6° approach as above, but after **5 s on the ground** the autopilot applies full throttle and
commands a 2 g pull-up.  Both the with-gear and body-collider-only scenarios are run.

Vertical markers: **dashed** = first touchdown, **solid** = go-around command issued.

In [ ]:
# ── Touch-and-go simulation ────────────────────────────────────────────────────
N_Z_ROTATE     = 2.0    # max commanded normal load factor during the go-around rotation
GAMMA_CLIMB_DEG = 6.0   # target climb flight-path angle for the go-around
T_SETTLE_S     = 0.5    # weight-on-wheels held this long before throttle (settled, not bouncing)
V_ROTATE_MPS   = 1.15 * V_STALL_MPS   # rotation speed: never rotate below this (no power-on stall)
T_END_TAG_S    = 60.0   # total duration (s) — enough to capture full cycle

def _run_touch_and_go(cfg, label=''):
    """Fly the stabilized FPA-hold approach, settle on the gear, full throttle, accelerate, then rotate gated on airspeed (not time)."""
    terrain_local = sim.FlatTerrain(TERRAIN_ELEV_M)
    ac_local = sim.Aircraft(json.dumps(cfg), dt_s=DT_S)
    ac_local.set_terrain(terrain_local)

    N_local = int(T_END_TAG_S / DT_S)
    r = dict(
        t_s       = np.zeros(N_local),
        agl_m     = np.zeros(N_local),
        alt_m     = np.zeros(N_local),
        vel_ned   = np.zeros((N_local, 3)),
        pitch_rad = np.zeros(N_local),
        alpha_rad = np.zeros(N_local),
        airspeed  = np.zeros(N_local),
        cf_force  = np.zeros((N_local, 3)),
        wow       = np.zeros(N_local, dtype=bool),
        throttle  = np.zeros(N_local),
        n_z_cmd   = np.zeros(N_local),
    )

    first_wow_t  = None
    settle_start = None
    powered      = False   # latched: full throttle, stays on through liftoff
    go_phase     = False   # latched: rotation, gated on airspeed
    go_t         = None    # time the go-around rotation was commanded
    gamma_target = abs(gamma_rad)
    gamma_climb  = math.radians(GAMMA_CLIMB_DEG)

    cmd_local = sim.AircraftCommand()
    cmd_local.n_y               = 0.0
    cmd_local.roll_rate_wind_rps = 0.0

    for i in range(N_local):
        s  = ac_local.state()
        cf = ac_local.contact_forces()
        t  = s.time_s
        v_h = math.hypot(s.velocity_north_mps, s.velocity_east_mps)

        if cf.weight_on_wheels and first_wow_t is None:
            first_wow_t = t
        # Settle detection: weight-on-wheels held continuously (not still bouncing).
        if cf.weight_on_wheels:
            if settle_start is None:
                settle_start = t
        else:
            settle_start = None
        # Sequenced touch-and-go (throttle and stick are NOT simultaneous): settle -> full throttle
        # (latched, stays on through liftoff) -> rotate once airspeed reaches V_ROTATE_MPS.
        if not powered and settle_start is not None and (t - settle_start) >= T_SETTLE_S:
            powered = True
        if powered and not go_phase and s.airspeed_m_s >= V_ROTATE_MPS:
            go_phase = True
            go_t = t

        cmd_local.throttle_nd = 1.0 if powered else 0.0
        if go_phase:
            # Rotate to and hold a climb flight-path angle (airspeed-gated, so never below stall).
            gamma_climb_cur = math.atan2(-s.velocity_down_mps, max(v_h, 0.1))   # climb-positive
            cmd_local.n_z = max(0.5, min(N_Z_ROTATE, 1.0 + 3.0 * (gamma_climb - gamma_climb_cur)))
        elif cf.weight_on_wheels:
            # On the ground, pre-rotation: hold (the settle term unloads the wing).
            cmd_local.n_z = 1.0
        else:
            # Airborne approach: flight-path-angle hold to the ground, no flare (same as _run_sim).
            gamma_cur = math.atan2(s.velocity_down_mps, max(v_h, 0.1))          # descent-positive
            cmd_local.n_z = max(0.2, min(1.8, math.cos(gamma_target) + FPA_GAIN * (gamma_cur - gamma_target)))

        r['t_s'][i]       = t
        r['agl_m'][i]     = ac_local.agl_m()
        r['alt_m'][i]     = s.altitude_m
        r['vel_ned'][i]   = [s.velocity_north_mps, s.velocity_east_mps, s.velocity_down_mps]
        r['pitch_rad'][i] = s.pitch_rad
        r['alpha_rad'][i] = s.alpha_rad
        r['airspeed'][i]  = s.airspeed_m_s
        r['cf_force'][i]  = cf.force_body_n
        r['wow'][i]       = cf.weight_on_wheels
        r['throttle'][i]  = cmd_local.throttle_nd
        r['n_z_cmd'][i]   = cmd_local.n_z

        ac_local.step(cmd_local, dt_s=DT_S, rho_kgm3=RHO_KGM3)

    r['td_t']  = first_wow_t
    r['go_t']  = go_t
    r['label'] = label
    return r


results_tag_gear = _run_touch_and_go(approach_cfg, label='Touch-and-go (gear)')

_cfg_tag_no_gear = copy.deepcopy(approach_cfg)
del _cfg_tag_no_gear['landing_gear']
results_tag_no_gear = _run_touch_and_go(_cfg_tag_no_gear, label='Touch-and-go (body collider only)')

for rr in [results_tag_gear, results_tag_no_gear]:
    td, go = rr['td_t'], rr['go_t']
    wow_times = rr['t_s'][rr['wow']]
    last_wow  = wow_times[-1] if wow_times.size else None
    if td is None:
        print(f'{rr["label"]}: no touchdown detected')
    else:
        lw_str = f'{last_wow:.3f} s' if last_wow is not None else 'none'
        td_str = f'{td:.3f} s' if td is not None else 'none'
        go_str = f'{go:.3f} s' if go is not None else 'none (never settled / below V_rotate)'
        print(f'{rr["label"]}: T/D = {td_str} | go-around = {go_str} | last WoW = {lw_str}')

In [ ]:
# ── Touch-and-go comparison plots ─────────────────────────────────────────────
_TAG_COLORS = {
    'Touch-and-go (gear)':               'tab:blue',
    'Touch-and-go (body collider only)': 'tab:orange',
}

fig_tag, axes_tag = plt.subplots(3, 2, figsize=(13, 11))
fig_tag.suptitle('Touch-and-Go — Gear vs Body Collider Only\n'
                 '(dashed = first T/D, solid = go-around command)',
                 fontsize=12, fontweight='bold')

for rr in [results_tag_gear, results_tag_no_gear]:
    col  = _TAG_COLORS[rr['label']]
    lbl  = rr['label']
    t    = rr['t_s']
    td_t = rr['td_t']
    go_t = rr['go_t']

    def _vlines(ax):
        if td_t: ax.axvline(td_t, color=col, lw=0.9, ls='--', alpha=0.6)
        if go_t: ax.axvline(go_t, color=col, lw=1.3, ls='-',  alpha=0.8)

    # AGL
    ax = axes_tag[0, 0]
    ax.plot(t, rr['agl_m'], color=col, label=lbl)
    _vlines(ax)

    # Airspeed
    ax = axes_tag[0, 1]
    ax.plot(t, rr['airspeed'], color=col, label=lbl)
    _vlines(ax)

    # Pitch
    ax = axes_tag[1, 0]
    ax.plot(t, np.degrees(rr['pitch_rad']), color=col, label=lbl)
    _vlines(ax)

    # WoW
    ax = axes_tag[1, 1]
    ax.plot(t, rr['wow'].astype(float), color=col, label=lbl, lw=1.2)
    _vlines(ax)

    # Contact normal force
    ax = axes_tag[2, 0]
    ax.plot(t, -rr['cf_force'][:, 2], color=col, label=lbl)
    _vlines(ax)

    # Throttle and n_z command
    ax = axes_tag[2, 1]
    ax.plot(t, rr['throttle'], color=col, lw=1.5, label=f'{lbl} — throttle')
    ax.plot(t, rr['n_z_cmd'],  color=col, lw=1.0, ls='--', label=f'{lbl} — n_z cmd')
    _vlines(ax)

# Decorate ────────────────────────────────────────────────────────────────────
ax = axes_tag[0, 0]
ax.axhline(0.0, color='saddlebrown', lw=1.5)
ax.set_xlabel('Time (s)'); ax.set_ylabel('AGL (m)')
ax.set_title('Height Above Ground (CG)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes_tag[0, 1]
ax.set_xlabel('Time (s)'); ax.set_ylabel('Airspeed (m/s)')
ax.set_title('Airspeed'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes_tag[1, 0]
ax.set_xlabel('Time (s)'); ax.set_ylabel('Pitch (°)')
ax.set_title('Pitch Angle'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes_tag[1, 1]
ax.set_ylim(-0.1, 1.3)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Weight-on-Wheels')
ax.set_title('Weight-on-Wheels Flag'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes_tag[2, 0]
ax.axhline(W_N, color='k', lw=0.8, ls='--', label=f'Weight {W_N:.1f} N')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Force (N)')
ax.set_title('Contact Normal Force (upward = −Fz_body)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes_tag[2, 1]
ax.set_xlabel('Time (s)'); ax.set_ylabel('Command value')
ax.set_title('Throttle (solid) and n_z Command (dashed)'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

fig_tag.tight_layout()
plt.show()

## Lateral-Asymmetry Landing Cases

Gear-present lateral-asymmetry touchdowns — the cases requested for landing-gear / ground-collider
validation, exercising the OQ-BC-12 **Alt B** contact-reaction model (persistent wind-axis roll rate,
Tustin stiff-stable integration, castering nose wheel, FBW `n_y`-loop yaw):

- **Banked** (~4° bank at touchdown) — one main touches first; the gear should right the airframe to
  wings-level and *stay* level (not spring back, not catapult).
- **Crabbed** (crosswind, `n_y = 0.15` ≈ 8° sideslip) — the aircraft decrabs; the **castering nose wheel
  produces no side force** (`F_y = 0`), so directional control is the FBW `n_y` model.
- **Banked + crabbed** — combined crosswind touchdown.

Each case is shown in its **own figure** (not overlaid with the gear-vs-no-gear comparison, which makes
these harder to read). Plots use the new **per-wheel `wheel_contact_diags()`** (lateral tyre force `F_y`,
slip angle) and the **`roll_rate_state_rps`** accessor.

> **Note:** a body-collider *involvement* trace within these gear cases (to see whether the wing/fuselage
> collider engages during a gear landing) is added once the collider-only accessor is bound (plan item
> IP-GCV-3). The current `contact_forces()` returns the summed gear+collider reaction.

In [ ]:
# ── Lateral-asymmetry landing runner ──────────────────────────────────────────
# Flies the same stabilized FPA-hold approach as _run_sim, but with a lateral setup:
#   • bank     — a wind-axis roll-rate command held for roll_dur_s, then released (the bank
#                holds through the descent, since the FBW rate-commands roll with no bank hold)
#   • sideslip — a steady lateral load-factor command n_y (an uncoordinated SKID; β ≠ 0, no crab)
#                held through touchdown to LATERALLY LOAD THE GEAR, then smoothly relaxed after
#                weight-on-wheels (a real decrab/rollout would center the directional input), and
#                finally a series of scripted n_y REVERSALS to exercise the model's ground-steering
#                response during roll-out. These reversals are open-loop (no controller); they are
#                authority-limited by the OQ-AC-6 Ny curvature limit as ground speed decays.
#   • crab     — a steady crosswind with the heading crabbed into it (β = 0, ground track stays
#                down-runway, heading offset by the crab). A crab is just the azimuth difference
#                between the air and ground velocities; it is set purely by ICs — NO controller.
# Gear present.  Collects the FULL longitudinal + lateral state, plus the per-wheel ContactDiag
# (F_y, F_z, slip), the OQ-BC-12 Alt B roll-rate state, the OQ-AC-4 crab angle, and the commanded n_y.

def _run_lateral(cfg, *, roll_rate=0.0, roll_dur_s=0.0, n_y=0.0, wind_e=0.0, agl0=2.0,
                 n_y_relax_tau_s=0.0, steer_amp=0.0, steer_period_s=4.0, steer_start_s=1.5,
                 label=''):
    terrain = sim.FlatTerrain(TERRAIN_ELEV_M)
    c = copy.deepcopy(cfg)
    vN = V_APPROACH_MPS * math.cos(abs(gamma_rad))
    vD = V_APPROACH_MPS * math.sin(abs(gamma_rad))
    hdg0 = math.atan2(-wind_e, vN)   # crab into the wind so the ground track stays down-runway (0 if calm)
    c['initial_state'] = {**approach_cfg['initial_state'],
                          'altitude_m':         TERRAIN_ELEV_M + agl0,
                          'velocity_north_mps': vN, 'velocity_east_mps': 0.0,
                          'velocity_down_mps':  vD, 'roll_rad': 0.0, 'heading_rad': hdg0,
                          'wind_north_mps': 0.0, 'wind_east_mps': wind_e, 'wind_down_mps': 0.0}
    wind_ned = [0.0, wind_e, 0.0]
    ac = sim.Aircraft(json.dumps(c), dt_s=DT_S)
    ac.set_terrain(terrain)
    cmd = sim.AircraftCommand()

    N = int(T_END_S / DT_S)
    scal = ('t_s', 'agl_m', 'alt_m', 'airspeed', 'ground_speed', 'roll', 'pitch', 'heading',
            'alpha', 'beta', 'crab', 'vE', 'roll_rate_state', 'p', 'q', 'r_yaw', 'cl_eff', 'wow',
            'cmd_ny')
    r = {k: np.zeros(N) for k in scal}
    for k in ('vel_ned', 'vel_body', 'accel_ned', 'cf_force', 'cf_moment',
              'Fy', 'Fz', 'alpha_t', 'strut', 'strut_rate', 'wheel_spd'):
        r[k] = np.zeros((N, 3))
    gamma_t = abs(gamma_rad)
    td_time = None   # touchdown time (first weight-on-wheels), set in the loop

    for i in range(N):
        s, cf = ac.state(), ac.contact_forces()
        t = s.time_s
        cmd.roll_rate_wind_rps = roll_rate if t <= roll_dur_s else 0.0
        # Directional (n_y) profile: hold the static skid to touchdown, then smoothly relax it and
        # add scripted ground-steering reversals during roll-out (open-loop — no controller).
        if cf.weight_on_wheels and td_time is None:
            td_time = t
        if td_time is None:
            ny_cmd = n_y
        else:
            dt_td  = t - td_time
            ny_cmd = n_y * math.exp(-dt_td / n_y_relax_tau_s) if n_y_relax_tau_s > 0.0 else 0.0
            if steer_amp and dt_td >= steer_start_s:
                ny_cmd += steer_amp * math.sin(2.0 * math.pi * (dt_td - steer_start_s) / steer_period_s)
        cmd.n_y = ny_cmd
        if cf.weight_on_wheels:
            cmd.n_z = 1.0
        else:
            vh = math.hypot(s.velocity_north_mps, s.velocity_east_mps)
            g_cur = math.atan2(s.velocity_down_mps, max(vh, 0.1))
            cmd.n_z = max(0.2, min(1.8, math.cos(gamma_t) + FPA_GAIN * (g_cur - gamma_t)))

        r['t_s'][i] = t;  r['agl_m'][i] = ac.agl_m();  r['alt_m'][i] = s.altitude_m
        r['airspeed'][i] = s.airspeed_m_s
        r['ground_speed'][i] = math.hypot(s.velocity_north_mps, s.velocity_east_mps)
        r['roll'][i] = math.degrees(s.roll_rad);   r['pitch'][i] = math.degrees(s.pitch_rad)
        r['heading'][i] = math.degrees(s.heading_rad)
        r['alpha'][i] = math.degrees(s.alpha_rad); r['beta'][i] = math.degrees(s.beta_rad)
        r['crab'][i] = math.degrees(s.crab_rad)
        r['cmd_ny'][i] = ny_cmd
        r['vE'][i] = s.velocity_east_mps
        r['vel_ned'][i]  = [s.velocity_north_mps, s.velocity_east_mps, s.velocity_down_mps]
        r['vel_body'][i] = s.velocity_body_mps
        r['accel_ned'][i] = s.acceleration_ned_mps2
        r['roll_rate_state'][i] = ac.roll_rate_state_rps
        r['p'][i] = math.degrees(s.rates_body_rad_s[0])
        r['q'][i] = math.degrees(s.rates_body_rad_s[1])
        r['r_yaw'][i] = math.degrees(s.rates_body_rad_s[2])
        r['cl_eff'][i] = ac.cl_eff
        r['cf_force'][i] = cf.force_body_n;  r['cf_moment'][i] = cf.moment_body_nm
        r['wow'][i] = cf.weight_on_wheels
        for j, d in enumerate(ac.wheel_contact_diags()):
            r['Fy'][i, j] = d.F_y; r['Fz'][i, j] = d.F_z
            r['alpha_t'][i, j] = math.degrees(d.alpha_t)
        for j, g in enumerate(ac.gear_strut_states()):
            r['strut'][i, j] = g.deflection_m
            r['strut_rate'][i, j] = g.deflection_rate_mps
            r['wheel_spd'][i, j] = g.wheel_speed_rps

        ac.step(cmd, dt_s=DT_S, rho_kgm3=RHO_KGM3, wind_ned_mps=wind_ned)

    r['north_pos'] = np.cumsum(r['vel_ned'][:, 0]) * DT_S
    r['east_pos']  = np.cumsum(r['vel_ned'][:, 1]) * DT_S
    r['td']    = r['t_s'][int(np.argmax(r['wow']))] if r['wow'].any() else None
    r['label'] = label
    return r


# Bank ≈ 4.3° (roll-rate 0.5 rad/s held 0.15 s, then released).  Sideslip: static n_y = 0.07 (a SKID,
# β ≈ 10°, no crab) held to touchdown, then RELAXED and followed by scripted n_y reversals that exercise
# the ground-steering response on roll-out (authority-limited by the OQ-AC-6 curvature limit as V decays).
# Crab: 1.4 m/s crosswind with a crabbed heading (β = 0, crab ≈ 9.5°). Sideslip vs crab stay DISTINCT
# cases (β-driven vs wind-driven).
LAT_CASES = [
    _run_lateral(approach_cfg, roll_rate=0.5, roll_dur_s=0.15,
                 label='Banked (~4°)'),
    # NOTE: scripted ground-steering reversals (steer_amp>0) currently DIVERGE — the on-ground
    # yaw response has no gear-aero moment balance yet (IP-CRB-11), so any lateral command on the
    # ground loops (even steer_amp=0.02 grows β to ~90°). Kept parametrized; enabled once IP-CRB-11
    # lands. For now the skid holds to touchdown (loads the gear) then relaxes — bounded roll-out.
    _run_lateral(approach_cfg, n_y=0.07, n_y_relax_tau_s=1.5, steer_amp=0.0,
                 label='Sideslip skid → relax after TD'),
    _run_lateral(approach_cfg, wind_e=1.4,
                 label='Crab (1.4 m/s crosswind, β=0)'),
    _run_lateral(approach_cfg, roll_rate=0.5, roll_dur_s=0.15, n_y=0.07, n_y_relax_tau_s=1.5,
                 label='Banked + sideslip'),
]

print(f'{"Case":40s} {"T/D":>6s} {"peak|roll|":>10s} {"|crab|air":>9s} {"|beta|air":>9s} '
      f'{"postTD|r|":>9s} {"maxAGLaft":>10s}')
for rr in LAT_CASES:
    td  = rr['td']
    wow = rr['wow'].astype(bool)
    air = ~wow
    after    = rr['agl_m'][int(np.argmax(rr['wow'])):].max() if wow.any() else float('nan')
    crab_air = np.abs(rr['crab'][air]).max() if air.any() else float('nan')
    beta_air = np.abs(rr['beta'][air]).max() if air.any() else float('nan')
    r_post   = np.abs(rr['r_yaw'][wow]).max() if wow.any() else float('nan')  # ground-steering response
    print(f'{rr["label"]:40s} {("-" if td is None else f"{td:5.2f}s"):>6s} '
          f'{np.abs(rr["roll"]).max():9.1f}° {crab_air:8.1f}° {beta_air:8.1f}° '
          f'{r_post:8.1f}° {after:9.2f}m')


In [ ]:
# ── Lateral-asymmetry plots — full longitudinal + lateral, one figure PER case ──
def _plot_lateral(rr):
    t, td = rr['t_s'], rr['td']

    def _v(a):
        if td is not None:
            a.axvline(td, color='red', ls='--', lw=0.9, alpha=0.6)
        a.grid(True, alpha=0.3); a.set_xlabel('Time (s)')

    # ---- Figure A: longitudinal & contact ----
    figA, ax = plt.subplots(3, 3, figsize=(15, 11))
    figA.suptitle(f"{rr['label']} — Longitudinal & Contact (gear present)",
                  fontsize=12, fontweight='bold')
    a = ax[0, 0]; a.plot(t, rr['alt_m'], label='alt'); a.plot(t, rr['agl_m'], label='AGL')
    a.axhline(0, color='saddlebrown', lw=1.2); a.set_ylabel('m')
    a.set_title('Altitude / AGL'); a.legend(fontsize=7); _v(a)
    a = ax[0, 1]; a.plot(t, rr['airspeed'], label='airspeed')
    a.plot(t, rr['ground_speed'], label='ground speed', ls='--')
    a.set_ylabel('m/s'); a.set_title('Speed'); a.legend(fontsize=7); _v(a)
    a = ax[0, 2]; a.plot(t, rr['pitch'], label='pitch θ'); a.plot(t, rr['alpha'], label='α', ls='--')
    a.set_ylabel('°'); a.set_title('Pitch / AoA'); a.legend(fontsize=7); _v(a)
    a = ax[1, 0]
    for k, lab in ((0, 'v_N'), (1, 'v_E'), (2, 'v_D')): a.plot(t, rr['vel_ned'][:, k], label=lab)
    a.set_ylabel('m/s'); a.set_title('NED velocity'); a.legend(fontsize=7); _v(a)
    a = ax[1, 1]
    for k, lab in ((0, 'u'), (1, 'v'), (2, 'w')): a.plot(t, rr['vel_body'][:, k], label=lab)
    a.set_ylabel('m/s'); a.set_title('Body velocity'); a.legend(fontsize=7); _v(a)
    a = ax[1, 2]; a.plot(t, -rr['cf_force'][:, 2], label='−Fz (up)')
    a.axhline(W_N, color='k', ls='--', lw=0.8, label=f'W={W_N:.0f} N')
    a.set_ylabel('N'); a.set_title('Contact normal force'); a.legend(fontsize=7); _v(a)
    a = ax[2, 0]
    for j, (n, c) in enumerate(zip(GEAR_NAMES, GEAR_COLORS)): a.plot(t, rr['strut'][:, j] * 1000, color=c, label=n)
    a.set_ylabel('mm'); a.set_title('Strut deflection'); a.legend(fontsize=7); _v(a)
    a = ax[2, 1]
    for j, (n, c) in enumerate(zip(GEAR_NAMES, GEAR_COLORS)): a.plot(t, rr['strut_rate'][:, j] * 1000, color=c, label=n)
    a.set_ylabel('mm/s'); a.set_title('Strut rate'); a.legend(fontsize=7); _v(a)
    a = ax[2, 2]
    for j, (n, c) in enumerate(zip(GEAR_NAMES, GEAR_COLORS)): a.plot(t, rr['wheel_spd'][:, j], color=c, label=n)
    a.set_ylabel('rad/s'); a.set_title('Wheel speed'); a.legend(fontsize=7); _v(a)
    figA.tight_layout(); plt.show()

    # ---- Figure B: lateral asymmetry ----
    figB, bx = plt.subplots(2, 3, figsize=(15, 8))
    figB.suptitle(f"{rr['label']} — Lateral Asymmetry (gear present)",
                  fontsize=12, fontweight='bold')
    a = bx[0, 0]; a.plot(t, rr['roll'], label='roll φ'); a.plot(t, rr['beta'], label='β', ls='--')
    a.plot(t, rr['heading'], label='heading', ls=':'); a.plot(t, rr['crab'], label='crab', ls='-.')
    a.set_ylabel('°'); a.set_title('Roll / Sideslip / Heading / Crab'); a.legend(fontsize=7); _v(a)
    a = bx[0, 1]; a.plot(t, np.degrees(rr['roll_rate_state']), label='ω_roll state (Alt B)')
    a.plot(t, rr['p'], label='p body', ls='--'); a.plot(t, rr['r_yaw'], label='r body', ls=':')
    a.set_ylabel('°/s'); a.set_title('Roll-rate state & body rates'); a.legend(fontsize=7); _v(a)
    a = bx[0, 2]; a.plot(rr['east_pos'], rr['north_pos'], 'k', lw=1.2)
    a.axvline(0.0, color='gray', ls=':', lw=1.0, label='runway CL')
    a.set_aspect('equal'); a.set_xlabel('East (m)'); a.set_ylabel('North (m)')
    a.set_title('Ground track (axis-equal)'); a.grid(True, alpha=0.3); a.legend(fontsize=7)
    a = bx[1, 0]
    for j, (n, c) in enumerate(zip(GEAR_NAMES, GEAR_COLORS)): a.plot(t, rr['Fy'][:, j], color=c, label=n)
    a.set_ylabel('N'); a.set_title('Per-wheel F_y (nose castering → 0)'); a.legend(fontsize=7); _v(a)
    a = bx[1, 1]; a.plot(t, (rr['strut'][:, 1] - rr['strut'][:, 2]) * 1000, 'purple')
    a.set_ylabel('mm'); a.set_title('Main strut L−R asymmetry'); _v(a)
    a = bx[1, 2]; a.plot(t, rr['cf_moment'][:, 0], label='Mx (roll)')
    a.plot(t, rr['cf_moment'][:, 2], label='Mz (yaw)')
    a.set_ylabel('N·m'); a.set_title('Contact moments'); a.legend(fontsize=7); _v(a)
    figB.tight_layout(); plt.show()


for _rr in LAT_CASES:
    _plot_lateral(_rr)
